Author: Will Blanton

# Imports

In [1]:
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import itertools
import sys
from pathlib import Path

from sentence_transformers import SentenceTransformer

from gensim.models.fasttext import load_facebook_vectors
from typing import List
from torch.utils.data import Dataset, DataLoader

import optuna
sys.path.insert(0, str(Path("src").resolve()))
from nyt_connections_solver.models import AttentionModel, Baseline, SetTransformer
from nyt_connections_solver.harness import SearchHarness
from nyt_connections_solver.evaluation import evaluate_puzzles, load_puzzles

# Graph attention experiment
Frozen Qwen text embeddings plus hashed character n-grams and structural spelling
features form the nodes. Shared edge providers compute semantic, English WordNet,
character, prefix/suffix, phrase, anagram, and spelling-structure channels.

Features are cached under `data/feature_cache/graph/`. Full-board tensors are reused
across all partial states. Set `REFRESH_FEATURE_CACHE = True` for a deliberate refresh;
provider versions, options, model revisions, and lexical fingerprints otherwise
invalidate only affected cache entries. Training remains an explicit notebook step.


In [2]:
from nyt_connections_solver.models import MultiheadAttentionBlock, SetDecoder

class GraphAttentionBlock(MultiheadAttentionBlock):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, attn_heads=heads, dropout=dropout)
        self.edge_bias_scale = float(edge_bias_scale)
        if self.edge_bias_scale < 0:
            raise ValueError("edge_bias_scale must be nonnegative")
        self.edge_projection = nn.Linear(edge_dim, heads, bias=False)
        # Start from ordinary attention and learn which relationships to use.
        nn.init.zeros_(self.edge_projection.weight)

    def attention_bias(self, edges):
        bias = self.edge_bias_scale * torch.tanh(self.edge_projection(edges))
        # (B, N, N, H) -> (B * H, N, N), as required by MultiheadAttention.
        diagonal = torch.eye(edges.shape[1], device=edges.device, dtype=torch.bool)
        bias = bias.masked_fill(diagonal[None, :, :, None], 0)
        return bias.permute(0, 3, 1, 2).reshape(-1, edges.shape[1], edges.shape[2])

    def forward(self, x, edges):
        # x: (B, N, D); edges: (B, N, N, E)
        bias = None if self.edge_bias_scale == 0 else self.attention_bias(edges).to(x.dtype)
        attended = self.attn(x, x, x, attn_mask=bias, need_weights=False)[0]
        x = self.h_norm(x + attended)
        return self.mab_norm(self.residual_proj(x) + self.ff(x))


class GraphEncoder(nn.Module):
    def __init__(self, width, depth, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__()
        self.mabs = nn.ModuleList([
            GraphAttentionBlock(width, heads, dropout, edge_dim, edge_bias_scale)
            for _ in range(depth)
        ])

    def forward(self, x, edges):
        for block in self.mabs:
            x = block(x, edges)
        return x


class GraphCrossAttentionBlock(MultiheadAttentionBlock):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, attn_heads=heads, dropout=dropout)
        self.edge_bias_scale = float(edge_bias_scale)
        self.edge_projection = nn.Linear(edge_dim, heads, bias=False)
        nn.init.zeros_(self.edge_projection.weight)

    def forward(self, seeds, words, word_edges):
        # seeds: (B * C, 1, D); words: (B * C, 4, D); word_edges: (B * C, 4, E)
        bias = None
        if self.edge_bias_scale != 0:
            bias = self.edge_bias_scale * torch.tanh(self.edge_projection(word_edges))
            bias = bias.permute(0, 2, 1).reshape(-1, 1, words.shape[1]).to(seeds.dtype)
        attended = self.attn(seeds, words, words, attn_mask=bias, need_weights=False)[0]
        x = self.h_norm(seeds + attended)
        return self.mab_norm(self.residual_proj(x) + self.ff(x))


class GraphSetDecoder(SetDecoder):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, width, num_seeds=1, mab_attn_heads=heads, dropout=dropout)
        original = self.mab
        self.mab = GraphCrossAttentionBlock(width, heads, dropout, edge_dim, edge_bias_scale)
        self.mab.load_state_dict(original.state_dict(), strict=False)

    def forward(self, x, edges):
        # x: (B * C, 4, D); edges: (B * C, 4, 4, E)
        words = self.ff1(x)
        seeds = self.seed_vectors.unsqueeze(0).expand(x.shape[0], -1, -1)
        # Diagonal edges are zero, so divide by the three other words.
        word_edges = edges.sum(dim=1) / (edges.shape[1] - 1)  # (B * C, 4, E)
        return self.mab(seeds, words, word_edges)


class GraphSetTransformer(SetTransformer):
    """Existing LTR scorer with edge-biased global and candidate attention."""
    def __init__(self, *, edge_dim=3, edge_bias_scale=1.0, **model_config):
        super().__init__(**model_config)
        self.edge_dim = edge_dim
        original = self.full_encoder
        first = original.mabs[0]
        self.full_encoder = GraphEncoder(
            first.attn.embed_dim, len(original.mabs), first.attn.num_heads,
            first.attn.dropout, edge_dim, edge_bias_scale,
        )
        # Preserve the original initialization; scale=0 is the same model.
        self.full_encoder.load_state_dict(original.state_dict(), strict=False)
        original = self.subset_encoder
        first = original.mabs[0]
        self.subset_encoder = GraphEncoder(
            first.attn.embed_dim, len(original.mabs), first.attn.num_heads,
            first.attn.dropout, edge_dim, edge_bias_scale,
        )
        self.subset_encoder.load_state_dict(original.state_dict(), strict=False)
        original = self.subset_decoder
        self.subset_decoder = GraphSetDecoder(
            original.mab.attn.embed_dim, original.mab.attn.num_heads,
            original.mab.attn.dropout, edge_dim, edge_bias_scale,
        )
        self.subset_decoder.load_state_dict(original.state_dict(), strict=False)

    def forward(self, x, group_idx=None, edge_features=None):
        B, N, _ = x.shape
        if edge_features is None or edge_features.shape != (B, N, N, self.edge_dim):
            raise ValueError("Expected edge_features with shape (B, N, N, edge_dim)")
        x = self.proj(self.input_norm(x))  # (B, N, D)
        x = self.full_encoder(x, edge_features)  # (B, N, D)
        idx = self.grouper.idx if group_idx is None else group_idx.to(x.device)
        x = self.grouper(x, group_idx=idx)  # (B, C, 4, D)
        B, C, group_size, D = x.shape
        # Select the same four indices on both edge axes for every candidate.
        candidate_edges = edge_features[:, idx[:, :, None], idx[:, None, :], :]
        candidate_edges = candidate_edges.reshape(B * C, group_size, group_size, self.edge_dim)
        x = x.reshape(B * C, group_size, D)
        x = self.subset_encoder(x, candidate_edges)  # (B * C, 4, D)
        x = self.subset_decoder(x, candidate_edges).squeeze(1)  # (B * C, D)
        return self.scorer(x.reshape(B, C, D))  # (B, C, 1)


def score_batch(model, x, batch, group_idx=None):
    kwargs = {}
    if isinstance(model, GraphSetTransformer):
        kwargs["edge_features"] = batch["edge_features"].to(x.device)
    return model(x, group_idx=group_idx, **kwargs)


def load_graph_harness(weights_path, device="auto", beam_width=20, state_batch_size=4,
                       cache_dir="data/feature_cache/graph"):
    """Reload this graph configuration; run imports and graph model definitions first."""
    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
    if checkpoint["model_class"] != "GraphSetTransformer" or "feature_metadata" not in checkpoint:
        raise ValueError("Expected a text/spelling graph checkpoint with feature metadata")
    resolved = ("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else device
    options = checkpoint["feature_metadata"]["options"]
    embedder = SentenceTransformer(checkpoint["embedding_model_name"],
                                  revision=options["revision"], device=str(resolved))
    nodes, edges, metadata = build_graph_features(
        embedder, checkpoint["embedding_model_name"], cache_dir=cache_dir, **options
    )
    if metadata != checkpoint["feature_metadata"] or list(edges.feature_names) != checkpoint["edge_feature_names"]:
        raise ValueError("Feature definitions or lexical resources differ from the checkpoint")
    config = checkpoint["model_config"]
    if config["in_dim"] != nodes.dim or config["edge_dim"] != edges.dim:
        raise ValueError("Checkpoint feature dimensions do not match")
    model = GraphSetTransformer(group_idx=torch.tensor([[0, 1, 2, 3]]), **config)
    model.load_state_dict(checkpoint["model_state_dict"])
    harness = SearchHarness.from_model(
        model, embedder, input_dim=nodes.dim, device=resolved,
        node_feature_builder=nodes, edge_feature_builder=edges.encode,
        beam_width=beam_width, state_batch_size=state_batch_size,
    )
    harness.checkpoint = checkpoint
    harness.weights_path = Path(weights_path)
    return harness


In [3]:
from pytorchltr.loss import LambdaNDCGLoss2
from pytorchltr.evaluation import ndcg

# Constants

In [4]:
DATASET_PATH = Path("data/connections.csv")
RELEVANCE_MODE = "binary"  # "binary" (exact match) or "graded" (maximum overlap)

M = 1820

# Helper Functions

In [5]:
def extract_features(word):
    return pd.DataFrame({word:{
        'length': len(word),
        'num_vowels': sum(c in 'aeiou' for c in word.lower()),
        'num_consonants': sum(c.isalpha() and c not in 'aeiou' for c in word.lower()),
        'ends_with_ing': int(word.lower().endswith('ing')),
        'ends_with_ed': int(word.lower().endswith('ed')),
        'is_palindrome': int(word.lower() == word[::-1].lower())
    }})

def extract_structural_features(words):
    """
    Extract basic structural features for the given words.
    """
    return torch.tensor(pd.concat([extract_features(w) for w in words], axis=1).T.values) 

In [6]:

def construct_groups_and_labels(num_sets_remaining=4, relevance_mode="binary"):
    """
    Produce candidate indices and binary or graded relevance labels.

    Labels assume the state's true groups are laid out consecutively in word order.
    """

    if num_sets_remaining < 1 or num_sets_remaining > 4:
        raise ValueError("num_sets_remaining must be between 1 and 4")
    if relevance_mode not in {"binary", "graded"}:
        raise ValueError("relevance_mode must be 'binary' or 'graded'")

    n_words = num_sets_remaining * 4
    groups = [set(range(i * 4, (i + 1) * 4)) for i in range(num_sets_remaining)]

    quads = list(itertools.combinations(range(n_words), 4))

    if relevance_mode == "binary":
        rel_scores = [int(set(q) in groups) for q in quads]
    else:
        rel_scores = [max(len(set(q).intersection(g)) for g in groups) for q in quads]

    return torch.tensor(quads), torch.tensor(rel_scores)

In [7]:
from nyt_connections_solver.data_loader import collate_keep_wordlists


# Create Dataset

Rerun from the top after changing `DATASET_PATH`; existing outputs may reflect an earlier dataset.

In [8]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

In [9]:
print(f"Cuda available: {torch.cuda.is_available()}")

Cuda available: True


In [10]:
# Multilingual frozen encoder (100+ languages); D is inferred below.
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
# Use symmetric, unprompted word embeddings in both training and the harness.
EMBEDDING_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"
word_embedder = SentenceTransformer(EMBEDDING_MODEL, revision=EMBEDDING_REVISION)
word_embedder.requires_grad_(False)
word_embedder.eval()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)

In [11]:
test_emb = word_embedder.encode("Hello")

In [12]:
D = len(test_emb)
D

1024

In [13]:
# ft = load_facebook_vectors('pretrained/cc.en.300.bin')

In [14]:
class FastTextEmbedder:
    """
    Wrap gensim embeddings for simpler conversion to tensors and to fit with sentence-transformers api for easy swapping
    """
    def __init__(self, ft_model):
        self.ft = ft_model
        self.dim = ft_model.vector_size

    def encode(self, word_list, convert_to_tensor=True):
        vectors = []
        for word in word_list:
            vectors.append(self.ft[word])

        vectors = np.stack(vectors)

        if convert_to_tensor:
            return torch.tensor(vectors, dtype=torch.float)
        else:
            return vectors

In [15]:
# word_embedder = FastTextEmbedder(ft)

In [16]:
connections_df = pd.read_csv(DATASET_PATH, index_col=0)
# connections_df.drop(columns='category', inplace=True)

# fix issues with incorrect format... (not worth going in and adding a mechanism to correct...)
connections_df.loc[1298, "connections"] = "['line', 'plane', 'point', 'solid']"
connections_df.loc[1892, "connections"] = "['abyss', 'fly', 'matrix', 'thing']"

# remove april fools samples since they include emojis or other potentially noisy samples
connections_df = connections_df[~connections_df['date'].str.contains("04-01")]
connections_df.reset_index(drop=True, inplace=True)

connections_df['connections'] = connections_df['connections'].apply(ast.literal_eval)
connections_df.head()

,date,category,connections
0,2023-06-12,palindromes,"[kayak, level, mom, race car]"
1,2023-06-12,keyboard keys,"[option, return, shift, tab]"
2,2023-06-12,nba teams,"[bucks, heat, jazz, nets]"
3,2023-06-12,wet weather,"[hail, rain, sleet, snow]"
4,2023-06-13,letter homophones,"[are, queue, sea, why]"


In [17]:
all_words = list(itertools.chain.from_iterable(
    connections_df
    .groupby('date')['connections']
    .agg(lambda lists: list(itertools.chain.from_iterable(lists)))
    .values
))
all_words[:10]

['kayak',
 'level',
 'mom',
 'race car',
 'option',
 'return',
 'shift',
 'tab',
 'bucks',
 'heat']

In [18]:
# [w for w in all_words if ('_' in w) or ('-' in w)]

In [19]:
# Character n-grams are hashed by TextSpellingFeatures; no fitted vocabulary is needed.
vectorizer = None


In [20]:
from nyt_connections_solver.graph_features import build_graph_features
from nyt_connections_solver.node_features import TextSpellingFeatures

FEATURE_CACHE_DIR = Path("data/feature_cache/graph")
REFRESH_FEATURE_CACHE = False
base_feature_builder, graph_edge_builder, feature_metadata = build_graph_features(
    word_embedder, EMBEDDING_MODEL, cache_dir=FEATURE_CACHE_DIR,
    refresh=REFRESH_FEATURE_CACHE, revision=EMBEDDING_REVISION,
)
graph_edge_providers = graph_edge_builder.providers
print("Node dimensions:", base_feature_builder.dim)
print("Edge channels:", graph_edge_builder.feature_names)


Node dimensions: 1165
Edge channels: ('semantic_cosine', 'sense_jaccard', 'rare_concept_jaccard', 'longest_common_substring', 'common_prefix', 'common_suffix', 'prefix_semantic_similarity', 'suffix_semantic_similarity', 'shared_preceding_word', 'shared_following_word', 'exact_anagram', 'character_multiset_overlap', 'both_palindromes', 'both_initial_double', 'both_consecutive_double_pairs')


In [21]:
from nyt_connections_solver.data_loader import GraphConnectionsDataset, make_state_loaders


In [22]:
idx, y = construct_groups_and_labels(relevance_mode=RELEVANCE_MODE)

In [23]:
idx[:10]

tensor([[ 0,  1,  2,  3],
        [ 0,  1,  2,  4],
        [ 0,  1,  2,  5],
        [ 0,  1,  2,  6],
        [ 0,  1,  2,  7],
        [ 0,  1,  2,  8],
        [ 0,  1,  2,  9],
        [ 0,  1,  2, 10],
        [ 0,  1,  2, 11],
        [ 0,  1,  2, 12]])

In [24]:
idx.unique()

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [25]:
y[:10]

tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [26]:
y.unique()

tensor([0, 1])

In [27]:
# split data
dates = connections_df["date"].unique()
train_idx = random.sample(range(len(dates)), k=int(len(dates) * .8))
train_dates = dates[train_idx]
test_dates = dates[~np.isin(range(len(dates)), train_idx)]

test_dates = np.random.permutation(test_dates)
split_idx = len(test_dates) // 2
val_dates = test_dates[split_idx:]
test_dates = test_dates[:split_idx]

print(f"Total Length: {len(dates)}")
print(f"Train length: {len(train_dates)}")
print(f"Val length: {len(val_dates)}")
print(f"Test length: {len(test_dates)}")

Total Length: 1210
Train length: 968
Val length: 121
Test length: 121


In [28]:
BATCH_SIZE = 16

# These prepared datasets/loaders are reused below; no duplicate preprocessing.
state_datasets = {}
state_loaders = {}
for split, dates_for_split in (("train", train_dates), ("val", val_dates), ("test", test_dates)):
    state_datasets[split], state_loaders[split] = make_state_loaders(
        connections_df.loc[connections_df["date"].isin(dates_for_split)],
        word_embedder, vectorizer, batch_size=BATCH_SIZE, shuffle=(split == "train"),
        feature_builder=base_feature_builder, edge_providers=graph_edge_providers,
        cache=graph_edge_builder.cache,
    )
train_state_datasets, val_state_datasets, test_state_datasets = (state_datasets[s] for s in ("train", "val", "test"))
train_state_loaders, val_state_loaders, test_state_loaders = (state_loaders[s] for s in ("train", "val", "test"))
train, val, test = (state_datasets[s][4] for s in ("train", "val", "test"))
train_loader, val_loader, test_loader = (state_loaders[s][4] for s in ("train", "val", "test"))


In [29]:
idx[0]

tensor([0, 1, 2, 3])

In [30]:
idx.shape

torch.Size([1820, 4])

In [31]:
batch = next(iter(val_loader))
print("Node features:", batch["x"].shape)
print("Edge features:", batch["edge_features"].shape)
print("Words:", batch["word_list"][0])


Node features: torch.Size([16, 16, 1165])
Edge features: torch.Size([16, 16, 16, 15])
Words: ['cray', 'jelly', 'silver', 'star', 'can', 'head', 'john', 'throne', 'calf', 'cub', 'joey', 'kid', 'ankle', 'knee', 'shin', 'thigh']


In [32]:
idx.shape

torch.Size([1820, 4])

# Model Architectures

## Model classes
The notebook uses shared `Baseline`, `AttentionModel`, and `SetTransformer`
architectures with an ordinary learned projection of the concatenated text and
spelling node features. The graph model additionally biases attention using the
named pairwise edge channels.


# Train Model

In [33]:
def topk_overlap(pred_scores, relevance, k=4):
    """
    Computes fraction of relevant items appearing in the model's top-k predictions.
    Args:
        pred_scores (torch.Tensor): (B, N) predicted scores
        relevance (torch.Tensor): (B, N) binary relevance labels (0 or 1)
        k (int): number of top items to consider
    Returns:
        torch.Tensor: mean overlap fraction across batch
    """
    _, pred_idx = torch.topk(pred_scores, k, dim=1)
    _, true_idx = torch.topk(relevance, k, dim=1)

    overlap = (pred_idx.unsqueeze(2) == true_idx.unsqueeze(1)).any(dim=2).float().sum(dim=1)
    return (overlap / k).mean()

In [34]:
def evaluate(model, loader, loss_fn, y, n_scalar, device, k=4, group_idx=None):
    model.eval()
    
    total_loss = 0
    total_ndcg = 0
    total_top4_acc = 0
    total_examples = 0

    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            
            B, _, D = x.shape
 
            scores = score_batch(model, x, batch, group_idx=group_idx)

            # expand to use in batch
            relevance = y.unsqueeze(0).expand(B, len(y)) # (B, 1820)

            n = n_scalar.expand(B)

            total_loss += loss_fn(scores, relevance, n).mean().item() * B
            total_ndcg += ndcg(scores, relevance, n, k=k).mean().item() * B
            total_top4_acc += topk_overlap(scores.squeeze(-1), relevance, k=k).item() * B
            total_examples += B

    if total_examples == 0:
        raise ValueError("Cannot evaluate an empty loader")

    avg_loss = total_loss / total_examples
    avg_ndcg = total_ndcg / total_examples
    avg_top4_acc = total_top4_acc / total_examples

    return avg_loss, avg_ndcg, avg_top4_acc

In [35]:

from tensorboardX import SummaryWriter
from datetime import datetime
def train(model, y, num_epochs, model_name, device, lr=1e-4):
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
    writer = SummaryWriter(log_dir=f"runs/{model_name}/connections_experiment_LTR-{current_time}")

    scaler = torch.amp.GradScaler(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    y = y.to(device)
    n_scalar = torch.tensor(M, device=device, dtype=torch.long)

    loss = LambdaNDCGLoss2().to(device)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_ndcg = 0
        train_top4_acc = 0

        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            x = batch["x"].to(device)
            
            B, _, D = x.shape
            
            # expand to use in batch
            relevance = y.unsqueeze(0).expand(B, len(y)) # (B, 1820)
            n = n_scalar.expand(B)

            # with torch.amp.autocast(dtype=torch.float16, device_type=str(device)):
            scores = model(x)
            batch_loss = loss(scores, relevance, n).mean()

            scaler.scale(batch_loss).backward()
            # TODO: check if gradient clipping is needed
            scaler.step(optimizer)
            scaler.update()

            # eval train
            with torch.no_grad():
                train_ndcg += ndcg(scores, relevance, n, k=4).mean().item()
                train_top4_acc += topk_overlap(scores.squeeze(-1), relevance, k=4).item()
                
            train_loss += batch_loss.item()

        train_loss /= len(train_loader)
        train_ndcg /= len(train_loader)
        train_top4_acc /= len(train_loader)

        val_loss, val_ndcg, val_top4_acc = evaluate(model, val_loader, loss, y, n_scalar, device)

        scheduler.step(val_loss)

        if epoch % 5 == 0 or epoch == num_epochs - 1:
            # --- Tensorboard logging ---
            writer.add_scalar("loss/train", train_loss, epoch)
            writer.add_scalar("loss/val", val_loss, epoch)

            writer.add_scalar("ndcg/train", train_ndcg, epoch)
            writer.add_scalar("ndcg/val", val_ndcg, epoch)

            writer.add_scalar("top4_acc/train", train_top4_acc, epoch)
            writer.add_scalar("top4_acc/val", val_top4_acc, epoch)

            current_lr = optimizer.param_groups[0]['lr']
            writer.add_scalar("lr", current_lr, epoch)

            for name, param in model.named_parameters():
                writer.add_histogram(f"param_hist/{name}", param, epoch)

                if param.grad is not None:
                    writer.add_histogram(f"grad_hist/{name}", param.grad, epoch)

            print(
                f"Epoch {epoch} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train NDCG: {train_ndcg:.4f} | "
                f"Train Top-4 Acc: {train_top4_acc:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val NDCG: {val_ndcg:.4f} | "
                f"Val Top-4 Acc: {val_top4_acc:.4f}"
            )
    
    return model.to('cpu')


def evaluate_full_state(model, loader, y, device, k=4):
    y = y.to(device)
    n_scalar = torch.tensor(len(y), device=device, dtype=torch.long)
    loss_fn = LambdaNDCGLoss2().to(device)
    return evaluate(model, loader, loss_fn, y, n_scalar, device, k=k)

In [36]:
def make_state_artifacts(device=None, relevance_mode="binary"):
    artifacts = {}
    for size in (2, 3, 4):
        group_idx, relevance = construct_groups_and_labels(size, relevance_mode=relevance_mode)
        if device is not None:
            group_idx = group_idx.to(device)
            relevance = relevance.to(device)

        artifacts[size] = {
            "idx": group_idx,
            "y": relevance,
            "n": torch.tensor(len(relevance), device=device, dtype=torch.long) if device is not None else torch.tensor(len(relevance), dtype=torch.long),
            "k": size,
        }
    return artifacts


def _next_batch(iterators, loaders, size):
    try:
        return next(iterators[size])
    except StopIteration:
        iterators[size] = iter(loaders[size])
        return next(iterators[size])


def evaluate_by_state_size(model, loaders, artifacts, loss_fn, device):
    metrics = {}
    total_examples = 0
    weighted_loss = 0
    weighted_ndcg = 0
    weighted_topk = 0

    for size, loader in loaders.items():
        artifact = artifacts[size]
        y = artifact["y"].to(device)
        n_scalar = artifact["n"].to(device)
        group_idx = artifact["idx"].to(device)
        k = artifact["k"]

        loss_value, ndcg_value, topk_value = evaluate(
            model,
            loader,
            loss_fn,
            y,
            n_scalar,
            device,
            k=k,
            group_idx=group_idx,
        )

        examples = len(loader.dataset)
        metrics[size] = {
            "loss": loss_value,
            "ndcg": ndcg_value,
            "topk_acc": topk_value,
            "examples": examples,
        }

        total_examples += examples
        weighted_loss += loss_value * examples
        weighted_ndcg += ndcg_value * examples
        weighted_topk += topk_value * examples

    metrics["weighted"] = {
        "loss": weighted_loss / total_examples,
        "ndcg": weighted_ndcg / total_examples,
        "topk_acc": weighted_topk / total_examples,
        "examples": total_examples,
    }
    return metrics


def format_state_metrics(prefix, metrics):
    parts = [
        f"{prefix} Loss: {metrics['weighted']['loss']:.4f}",
        f"{prefix} NDCG: {metrics['weighted']['ndcg']:.4f}",
        f"{prefix} Top-k Acc: {metrics['weighted']['topk_acc']:.4f}",
    ]
    for size in (2, 3, 4):
        if size in metrics:
            parts.append(f"{prefix} s{size} NDCG: {metrics[size]['ndcg']:.4f}")
            parts.append(f"{prefix} s{size} Top-{size}: {metrics[size]['topk_acc']:.4f}")
    return " | ".join(parts)


def train_curriculum_mixed(
    model,
    train_loaders,
    val_loaders,
    artifacts,
    phases,
    model_name,
    device,
    lr=5e-5,
    weight_decay=1e-5,
    steps_per_epoch=None,
    selection_size=None,
):
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
    writer = SummaryWriter(log_dir=f"runs/{model_name}/connections_curriculum_LTR-{current_time}")

    model = model.to(device)
    loss_fn = LambdaNDCGLoss2().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    scaler = torch.amp.GradScaler(device)

    artifacts = {
        size: {key: value.to(device) if torch.is_tensor(value) else value for key, value in artifact.items()}
        for size, artifact in artifacts.items()
    }

    iterators = {size: iter(loader) for size, loader in train_loaders.items()}
    best_val_ndcg = -float("inf")
    best_state_dict = None
    global_epoch = 0

    for phase in phases:
        active_sizes = phase["sizes"]
        weights = phase.get("weights", [1 / len(active_sizes)] * len(active_sizes))
        phase_steps = steps_per_epoch or max(len(train_loaders[size]) for size in active_sizes)

        for phase_epoch in range(phase["epochs"]):
            model.train()
            train_loss = 0
            train_ndcg = 0
            train_topk_acc = 0

            for _ in range(phase_steps):
                size = random.choices(active_sizes, weights=weights, k=1)[0]
                artifact = artifacts[size]
                batch = _next_batch(iterators, train_loaders, size)

                optimizer.zero_grad(set_to_none=True)
                x = batch["x"].to(device)
                B, _, _ = x.shape

                y = artifact["y"]
                group_idx = artifact["idx"]
                n = artifact["n"].expand(B)
                relevance = y.unsqueeze(0).expand(B, len(y))

                scores = score_batch(model, x, batch, group_idx=group_idx)
                batch_loss = loss_fn(scores, relevance, n).mean()

                scaler.scale(batch_loss).backward()
                scaler.step(optimizer)
                scaler.update()

                with torch.no_grad():
                    train_ndcg += ndcg(scores, relevance, n, k=artifact["k"]).mean().item()
                    train_topk_acc += topk_overlap(scores.squeeze(-1), relevance, k=artifact["k"]).item()
                    train_loss += batch_loss.item()

            train_loss /= phase_steps
            train_ndcg /= phase_steps
            train_topk_acc /= phase_steps

            val_active = {size: val_loaders[size] for size in active_sizes}
            val_metrics = evaluate_by_state_size(model, val_active, artifacts, loss_fn, device)

            selection_key = "weighted" if selection_size is None else selection_size
            if selection_key not in val_metrics:
                raise ValueError(f"Selection state size {selection_size} is not active in phase {phase['name']}")
            selection_metrics = val_metrics[selection_key]
            scheduler.step(selection_metrics["loss"])

            if selection_metrics["ndcg"] > best_val_ndcg:
                best_val_ndcg = selection_metrics["ndcg"]
                best_state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

            writer.add_scalar("loss/train", train_loss, global_epoch)
            writer.add_scalar("ndcg/train", train_ndcg, global_epoch)
            writer.add_scalar("topk_acc/train", train_topk_acc, global_epoch)
            writer.add_scalar("loss/val_weighted", val_metrics["weighted"]["loss"], global_epoch)
            writer.add_scalar("ndcg/val_weighted", val_metrics["weighted"]["ndcg"], global_epoch)
            writer.add_scalar("topk_acc/val_weighted", val_metrics["weighted"]["topk_acc"], global_epoch)

            for size in active_sizes:
                writer.add_scalar(f"ndcg/val_size_{size}", val_metrics[size]["ndcg"], global_epoch)
                writer.add_scalar(f"topk_acc/val_size_{size}", val_metrics[size]["topk_acc"], global_epoch)

            if phase_epoch % 5 == 0 or phase_epoch == phase["epochs"] - 1:
                message = (
                    f"Phase {phase['name']} Epoch {phase_epoch} | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Train NDCG: {train_ndcg:.4f} | "
                    f"Train Top-k Acc: {train_topk_acc:.4f} | "
                    f"{format_state_metrics('Val', val_metrics)}"
                )
                print(message)

            global_epoch += 1

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model.to('cpu')


def evaluate_state_ndcg(model, loaders, artifacts, device):
    """Ranking-only validation: no quadratic pairwise loss and no search."""
    model.eval()
    metrics = {}
    with torch.inference_mode():
        for size, loader in loaders.items():
            artifact = artifacts[size]
            total, count = 0.0, 0
            for batch in loader:
                x = batch["x"].to(device)
                scores = score_batch(model, x, batch, group_idx=artifact["idx"].to(device))
                relevance = artifact["y"].to(device).unsqueeze(0).expand(len(x), -1)
                lengths = artifact["n"].to(device).expand(len(x))
                total += ndcg(scores, relevance, lengths, k=size).sum().item()
                count += len(x)
            if not count:
                raise ValueError("Cannot validate an empty loader")
            metrics[size] = total / count
    return metrics


def train_mixed_states(
    model, train_loaders, val_loaders, artifacts, epochs, device,
    lr, weight_decay, trial=None, steps_per_epoch=None,
    early_stopping_patience=6, min_delta=1e-4, min_lr=1e-6,
):
    """Mixed-state LTR with NDCG selection, early stopping, and best-weight restore."""
    from time import perf_counter
    if epochs < 1 or early_stopping_patience < 1 or min_delta < 0:
        raise ValueError("Invalid epoch, patience, or improvement setting")
    if not 0 <= min_lr <= lr:
        raise ValueError("min_lr must be between zero and the initial learning rate")
    model = model.to(device)
    loss_fn = LambdaNDCGLoss2().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2, threshold=min_delta,
        threshold_mode="abs", min_lr=min_lr,
    )
    scaler = torch.amp.GradScaler(torch.device(device).type, enabled=torch.device(device).type == "cuda")
    artifacts = {
        size: {key: value.to(device) if torch.is_tensor(value) else value for key, value in item.items()}
        for size, item in artifacts.items()
    }
    sizes = tuple(sorted(train_loaders))
    if set(val_loaders) != set(sizes):
        raise ValueError("Training and validation must use the same state sizes")
    iterators = {size: iter(loader) for size, loader in train_loaders.items()}
    # One epoch has the full-board loader's update budget, not the six-times-larger
    # two-group loader's length. Sampling still mixes all state sizes uniformly.
    if steps_per_epoch is None:
        steps_per_epoch = len(train_loaders[max(sizes)])
    if steps_per_epoch < 1:
        raise ValueError("steps_per_epoch must be positive")
    best_val_ndcg = patience_reference = -float("inf")
    best_state_dict = None
    stale_epochs, best_epoch = 0, 0
    history = []

    for epoch in range(epochs):
        started = perf_counter()
        model.train()
        train_loss = 0.0
        for _ in range(steps_per_epoch):
            size = random.choice(sizes)
            batch = _next_batch(iterators, train_loaders, size)
            artifact = artifacts[size]
            x = batch["x"].to(device)
            relevance = artifact["y"].unsqueeze(0).expand(len(x), -1)
            lengths = artifact["n"].expand(len(x))
            optimizer.zero_grad(set_to_none=True)
            scores = score_batch(model, x, batch, group_idx=artifact["idx"])
            batch_loss = loss_fn(scores, relevance, lengths).mean()
            scaler.scale(batch_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += batch_loss.item()
        train_seconds = perf_counter() - started
        validation_started = perf_counter()
        metrics = evaluate_state_ndcg(model, val_loaders, artifacts, device)
        val_ndcg = sum(metrics.values()) / len(metrics)
        if not np.isfinite(val_ndcg):
            raise ValueError("Nonfinite validation NDCG")
        validation_seconds = perf_counter() - validation_started
        scheduler.step(val_ndcg)
        if val_ndcg > best_val_ndcg:
            best_val_ndcg, best_epoch = val_ndcg, epoch + 1
            best_state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if val_ndcg > patience_reference + min_delta:
            patience_reference, stale_epochs = val_ndcg, 0
        else:
            stale_epochs += 1
        history.append({"epoch": epoch + 1, "train_seconds": train_seconds,
                        "validation_seconds": validation_seconds, "val_ndcg": val_ndcg,
                        "lr": optimizer.param_groups[0]["lr"]})
        if trial is not None:
            trial.report(val_ndcg, epoch)
            trial.set_user_attr("epochs_ran", epoch + 1)
            trial.set_user_attr("best_epoch", best_epoch)
            trial.set_user_attr("best_validation_ndcg", best_val_ndcg)
            if trial.should_prune():
                raise optuna.TrialPruned()
        print(
            f"Epoch {epoch + 1}/{epochs} | Loss: {train_loss / steps_per_epoch:.4f} | "
            f"Val mean state NDCG: {val_ndcg:.4f} | LR: {optimizer.param_groups[0]['lr']:.2g} | "
            f"Train: {train_seconds:.1f}s | Validation: {validation_seconds:.1f}s"
        )
        if stale_epochs >= early_stopping_patience:
            print(f"Early stopping; restoring epoch {best_epoch} (NDCG {best_val_ndcg:.4f}).")
            break
    model.load_state_dict(best_state_dict)
    model.training_summary = {"best_validation_ndcg": best_val_ndcg,
                              "best_epoch": best_epoch, "epochs_ran": len(history),
                              "steps_per_epoch": steps_per_epoch, "history": history}
    return model.to("cpu")


In [37]:
RUN_BASELINES = False  # optional comparisons; not required for graph tuning
EPOCHS = 100
M = 1820
BASELINE_EPOCHS = 50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state_artifacts = make_state_artifacts(relevance_mode=RELEVANCE_MODE)
cpu_device = torch.device("cpu")
cpu_artifacts = make_state_artifacts(device=cpu_device, relevance_mode=RELEVANCE_MODE)
ranking_loss = LambdaNDCGLoss2().to(cpu_device)


## Learning diagnostic (run before Optuna)

Set `RUN_LEARNING_DIAGNOSTIC = True` and run the following cell after preparing the
features, datasets, and training helpers above. It trains one fresh graph model
with dropout 0, LR 3e-4, zero weight decay, and the existing LTR loss for a fixed
1,000 updates. There is no LR decay, early stopping, pruning, or search.

The run reports initial and subsequent training/validation NDCG separately for
2/3/4 remaining sets on fixed samples, plus per-size training losses in
`diagnostic_history`. Training samples the full training datasets with balanced
state-size update counts. No test data are used. `diagnostic_model` is separate
from the Optuna-selected `model`; this cell does not start the sweep. Run this
cell on its own and inspect its results before running the Optuna cell below.


In [41]:
def run_learning_diagnostic(total_updates=1000, eval_every=100, sample_states=128,
                            lr=3e-4, seed=0):
    """Fixed-budget LTR run; no scheduler, early stopping, Optuna, or test split."""
    from time import perf_counter
    from torch.utils.data import Subset
    if total_updates < 1 or eval_every < 1 or sample_states < 1:
        raise ValueError("Update budget, evaluation interval, and sample size must be positive")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    size_rng = random.Random(seed)
    sizes = (2, 3, 4)

    def fixed_loaders(datasets):
        loaders = {}
        for size in sizes:
            dataset = datasets[size]
            indices = random.Random(seed + size).sample(
                range(len(dataset)), min(sample_states, len(dataset))
            )
            loaders[size] = DataLoader(
                Subset(dataset, sorted(indices)), batch_size=BATCH_SIZE,
                shuffle=False, collate_fn=collate_keep_wordlists,
            )
        return loaders

    # These samples stay identical for every measurement and use no test data.
    train_diagnostic_loaders = fixed_loaders(train_state_datasets)
    val_diagnostic_loaders = fixed_loaders(val_state_datasets)
    diagnostic_model = GraphSetTransformer(
        in_dim=base_feature_builder.dim, edge_dim=graph_edge_builder.dim,
        edge_bias_scale=0.5, group_idx=state_artifacts[4]["idx"],
        lin_layers=2, lin_width=512, attn_layers=1, attn_width=256,
        attn_heads=8, dropout=0.0,
    ).to(device)
    optimizer = torch.optim.Adam(diagnostic_model.parameters(), lr=lr, weight_decay=0.0)
    loss_fn = LambdaNDCGLoss2().to(device)
    artifacts = {
        size: {key: value.to(device) if torch.is_tensor(value) else value for key, value in item.items()}
        for size, item in state_artifacts.items()
    }
    # Separate generators prevent diagnostic evaluation from advancing the
    # training shuffle RNG. Training still uses the full training datasets.
    training_loaders = {
        size: DataLoader(train_state_datasets[size], batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=collate_keep_wordlists,
                         generator=torch.Generator().manual_seed(seed + size))
        for size in sizes
    }
    iterators = {size: iter(loader) for size, loader in training_loaders.items()}
    history = []
    updates_by_size = {size: 0 for size in sizes}
    losses = {size: [] for size in sizes}
    training_seconds = 0.0

    def measure(step):
        started = perf_counter()
        train_metrics = evaluate_state_ndcg(diagnostic_model, train_diagnostic_loaders, artifacts, device)
        val_metrics = evaluate_state_ndcg(diagnostic_model, val_diagnostic_loaders, artifacts, device)
        row = {"updates": step, "train_seconds": training_seconds,
               "validation_seconds": perf_counter() - started, "lr": lr}
        parts = []
        for size in sizes:
            row[f"train_ndcg_s{size}"] = train_metrics[size]
            row[f"val_ndcg_s{size}"] = val_metrics[size]
            row[f"updates_s{size}"] = updates_by_size[size]
            row[f"train_loss_s{size}"] = float(np.mean(losses[size])) if losses[size] else float("nan")
            parts.append(f"s{size} train/val: {train_metrics[size]:.4f}/{val_metrics[size]:.4f}")
            losses[size].clear()
        history.append(row)
        print(f"Update {step}/{total_updates} | " + " | ".join(parts)
              + f" | Train: {training_seconds:.1f}s | Evaluation: {row['validation_seconds']:.1f}s")

    measure(0)  # distinguish actual learning from a favorable initialization
    pending_sizes = []
    for step in range(1, total_updates + 1):
        if not pending_sizes:
            pending_sizes = list(sizes)
            size_rng.shuffle(pending_sizes)
        size = pending_sizes.pop()
        started = perf_counter()
        diagnostic_model.train()
        batch = _next_batch(iterators, training_loaders, size)
        artifact = artifacts[size]
        x = batch["x"].to(device)
        relevance = artifact["y"].unsqueeze(0).expand(len(x), -1)
        lengths = artifact["n"].expand(len(x))
        optimizer.zero_grad(set_to_none=True)
        scores = score_batch(diagnostic_model, x, batch, group_idx=artifact["idx"])
        loss = loss_fn(scores, relevance, lengths).mean()
        if not torch.isfinite(loss):
            raise ValueError(f"Nonfinite diagnostic loss at update {step}")
        loss.backward()
        optimizer.step()
        losses[size].append(loss.item())
        updates_by_size[size] += 1
        training_seconds += perf_counter() - started
        if step % eval_every == 0 or step == total_updates:
            measure(step)
            training_seconds = 0.0
    diagnostic_model.eval().cpu()
    return diagnostic_model, pd.DataFrame(history)


RUN_LEARNING_DIAGNOSTIC = True
DIAGNOSTIC_UPDATES = 1000
DIAGNOSTIC_EVAL_EVERY = 100
if RUN_LEARNING_DIAGNOSTIC:
    diagnostic_model, diagnostic_history = run_learning_diagnostic(
        total_updates=DIAGNOSTIC_UPDATES, eval_every=DIAGNOSTIC_EVAL_EVERY,
    )
    display(diagnostic_history)


Update 0/1000 | s2 train/val: 0.0229/0.0234 | s3 train/val: 0.0078/0.0092 | s4 train/val: 0.0061/0.0060 | Train: 0.0s | Evaluation: 0.5s
Update 100/1000 | s2 train/val: 0.1190/0.1038 | s3 train/val: 0.0340/0.0230 | s4 train/val: 0.0166/0.0099 | Train: 5.1s | Evaluation: 0.5s
Update 200/1000 | s2 train/val: 0.1616/0.1633 | s3 train/val: 0.0657/0.0717 | s4 train/val: 0.0468/0.0264 | Train: 4.8s | Evaluation: 0.5s
Update 300/1000 | s2 train/val: 0.2006/0.1850 | s3 train/val: 0.1056/0.0740 | s4 train/val: 0.0799/0.0461 | Train: 5.1s | Evaluation: 0.5s
Update 400/1000 | s2 train/val: 0.2818/0.2011 | s3 train/val: 0.1754/0.0877 | s4 train/val: 0.1116/0.0384 | Train: 5.0s | Evaluation: 0.5s
Update 500/1000 | s2 train/val: 0.2810/0.2316 | s3 train/val: 0.2109/0.0813 | s4 train/val: 0.1280/0.0543 | Train: 5.0s | Evaluation: 0.5s
Update 600/1000 | s2 train/val: 0.3718/0.2168 | s3 train/val: 0.2678/0.0859 | s4 train/val: 0.1417/0.0453 | Train: 4.9s | Evaluation: 0.5s
Update 700/1000 | s2 train/va

,updates,train_seconds,validation_seconds,lr,train_ndcg_s2,val_ndcg_s2,updates_s2,train_loss_s2,train_ndcg_s3,val_ndcg_s3,updates_s3,train_loss_s3,train_ndcg_s4,val_ndcg_s4,updates_s4,train_loss_s4
0,0,0.000000,0.526024,0.0003,0.022924,0.023438,0,NaN,0.007813,0.009166,0,NaN,0.006100,0.006005,0,NaN
1,100,5.061348,0.521304,0.0003,0.118955,0.103844,33,0.970098,0.034043,0.022957,33,1.364466,0.016635,0.009878,34,1.615287
2,200,4.804371,0.514831,0.0003,0.161554,0.163322,67,0.897303,0.065686,0.071666,67,1.321705,0.046783,0.026431,66,1.581299
3,300,5.054833,0.521359,0.0003,0.200616,0.184991,100,0.848619,0.105622,0.073979,100,1.273418,0.079885,0.046064,100,1.551897
4,400,5.010608,0.521629,0.0003,0.281764,0.201130,133,0.788663,0.175367,0.087684,133,1.215044,0.111650,0.038442,134,1.506698
5,500,4.967977,0.523973,0.0003,0.281023,0.231639,167,0.718988,0.210851,0.081311,166,1.166566,0.128028,0.054350,167,1.464606
6,600,4.907556,0.521470,0.0003,0.371750,0.216755,200,0.689612,0.267764,0.085937,200,1.125942,0.141706,0.045296,200,1.435213
7,700,5.066771,0.519569,0.0003,0.436532,0.226849,233,0.634360,0.319485,0.103222,233,1.074873,0.207725,0.057973,234,1.377456
8,800,4.902240,0.521819,0.0003,0.519961,0.221031,266,0.553485,0.434620,0.075332,267,0.998141,0.290079,0.047454,267,1.309493
9,900,4.953365,0.520465,0.0003,0.531823,0.219036,300,0.536208,0.455090,0.099642,300,0.907668,0.355554,0.054227,300,1.261898


## Baseline Model

Baseline models train on the same uniformly sampled mixture of 2-, 3-, and 4-group states as the Set Transformer. Their architectures and learning rates remain fixed; no baseline hyperparameter search is run. 

In [38]:
if RUN_BASELINES:
    model = Baseline(
        in_dim=base_feature_builder.dim,
        group_idx=idx,
        layers=[512, 512, 512]
    ).to(device)

    model = train_mixed_states(
        model, train_state_loaders, val_state_loaders, state_artifacts,
        epochs=BASELINE_EPOCHS, device=device, lr=1e-4, weight_decay=1e-5,
    )
    baseline_val_metrics = evaluate_by_state_size(model, val_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    baseline_test_metrics = evaluate_by_state_size(model, test_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    print(format_state_metrics("Baseline Val", baseline_val_metrics))
    print(format_state_metrics("Baseline Test", baseline_test_metrics))

## Attention Model

This model uses the same multi-size data mixture and validation selection as the MLP baseline, with a fixed architecture and learning rate. 

In [39]:
if RUN_BASELINES:

    model = AttentionModel(
        in_dim=base_feature_builder.dim,
        group_idx=idx,
        layers=[1024, 512, 256],
        attn_layers=5
    ).to(device)

    model = train_mixed_states(
        model, train_state_loaders, val_state_loaders, state_artifacts,
        epochs=BASELINE_EPOCHS, device=device, lr=5e-5, weight_decay=1e-5,
    )
    attention_val_metrics = evaluate_by_state_size(model, val_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    attention_test_metrics = evaluate_by_state_size(model, test_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    print(format_state_metrics("Attention Val", attention_val_metrics))
    print(format_state_metrics("Attention Test", attention_test_metrics))

## Graph Set Transformer

Optuna selects by mean validation NDCG across 2/3/4 remaining sets. Trials use a
fixed validation subset, at most 25 epochs, and six-epoch plateau early stopping.
Each epoch uses the full-board loader's batch count while sampling all state sizes.
Training keeps the existing LTR loss; validation computes NDCG without that loss.
Timings separate training and validation. Search is disabled during tuning and is
optional after selection with `RUN_FINAL_SEARCH`. Full ranking metrics are reported
once for the selected model. These shorter runs are a tuning budget, not equivalent
to the previous 50 long epochs. Increase the budget explicitly for longer runs.


In [40]:
OPTUNA_TRIALS = 12
OPTUNA_EPOCHS = 25  # maximum; plateaued trials stop sooner
TUNING_PATIENCE = 6
TUNING_MIN_DELTA = 1e-4
TUNING_MIN_LR = 1e-6
TUNING_STEPS_PER_EPOCH = len(train_state_loaders[4])
TUNING_VAL_MAX_STATES_PER_SIZE = 128  # fixed, evenly spread subset; None uses all
EDGE_BIAS_SCALES = [0.25, 0.5, 1.0, 2.0]
TUNING_SEED = 0
RUN_FINAL_SEARCH = False
FINAL_BEAM_WIDTH = 20

from torch.utils.data import Subset

def tuning_validation_loaders(datasets, max_states, batch_size):
    if max_states is not None and max_states < 1:
        raise ValueError("max_states must be positive or None")
    loaders = {}
    for size, dataset in datasets.items():
        count = len(dataset) if max_states is None else min(len(dataset), max_states)
        indices = np.linspace(0, len(dataset) - 1, count, dtype=int).tolist()
        loaders[size] = DataLoader(Subset(dataset, indices), batch_size=batch_size,
                                  shuffle=False, collate_fn=collate_keep_wordlists)
    return loaders

tuning_val_loaders = tuning_validation_loaders(
    val_state_datasets, TUNING_VAL_MAX_STATES_PER_SIZE, BATCH_SIZE
)
retained_trial = {"score": -float("inf")}

def objective(trial):
    random.seed(TUNING_SEED)
    np.random.seed(TUNING_SEED)
    torch.manual_seed(TUNING_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TUNING_SEED)

    model_config = {
        "edge_dim": graph_edge_builder.dim,
        "edge_bias_scale": trial.suggest_categorical("edge_bias_scale", EDGE_BIAS_SCALES),
        "in_dim": base_feature_builder.dim,
        "lin_layers": trial.suggest_int("lin_layers", 2, 3),
        "lin_width": trial.suggest_categorical("lin_width", [512, 768, 1024]),
        "attn_layers": 1,
        "attn_width": trial.suggest_categorical("attn_width", [256, 384, 512]),
        "attn_heads": 8,
        "dropout": trial.suggest_float("dropout", 0.1, 0.25),
    }
    lr = trial.suggest_float("lr", 3e-5, 2e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    trial_model = GraphSetTransformer(
        group_idx=state_artifacts[4]["idx"], **model_config
    ).to(device)
    try:
        trial_model = train_mixed_states(
            trial_model, train_state_loaders, tuning_val_loaders, state_artifacts,
            epochs=OPTUNA_EPOCHS, device=device, lr=lr, weight_decay=weight_decay,
            trial=trial, steps_per_epoch=TUNING_STEPS_PER_EPOCH,
            early_stopping_patience=TUNING_PATIENCE, min_delta=TUNING_MIN_DELTA,
            min_lr=TUNING_MIN_LR,
        )
        summary = trial_model.training_summary
        score = summary["best_validation_ndcg"]
        trial.set_user_attr("training_summary", summary)
        # Keep only the best completed model, not one large checkpoint per trial.
        if score > retained_trial["score"]:
            retained_trial.update(score=score, number=trial.number, config=model_config,
                                  state={k: v.detach().cpu().clone() for k, v in trial_model.state_dict().items()})
        return score
    finally:
        trial_model.to("cpu")
        if torch.device(device).type == "cuda":
            torch.cuda.empty_cache()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=3),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
assert retained_trial["number"] == best_trial.number
best_model_config = retained_trial["config"]
best_training_config = {
    "relevance_mode": RELEVANCE_MODE,
    "selection_metric": "mean_state_ndcg",
    "max_epochs": OPTUNA_EPOCHS,
    "early_stopping_patience": TUNING_PATIENCE,
    "min_delta": TUNING_MIN_DELTA,
    "min_lr": TUNING_MIN_LR,
    "steps_per_epoch": TUNING_STEPS_PER_EPOCH,
    "validation_max_states_per_size": TUNING_VAL_MAX_STATES_PER_SIZE,
    "training_summary": best_trial.user_attrs["training_summary"],
    "lr": best_trial.params["lr"],
    "weight_decay": best_trial.params["weight_decay"],
}
model = GraphSetTransformer(group_idx=state_artifacts[4]["idx"], **best_model_config)
model.load_state_dict(retained_trial["state"])
model = model.to("cpu").eval()
print("Best tuning validation NDCG:", best_trial.value)
print("Best epoch:", best_trial.user_attrs["best_epoch"])
optuna_results = study.trials_dataframe()
optuna_results

# Full ranking evaluation once after selection, on the configured device.
model = model.to(device).eval()
final_val_ndcg = evaluate_state_ndcg(model, val_state_loaders, state_artifacts, device)
final_test_ndcg = evaluate_state_ndcg(model, test_state_loaders, state_artifacts, device)
print("Full validation NDCG by remaining sets:", final_val_ndcg)
print("Full test NDCG by remaining sets:", final_test_ndcg)

# Search is optional and never part of the Optuna objective.
if RUN_FINAL_SEARCH:
    test_puzzles = load_puzzles(
        DATASET_PATH, pd.to_datetime(test_dates).strftime("%Y-%m-%d").tolist()
    )
    test_harness = SearchHarness.from_model(
        model, word_embedder, input_dim=base_feature_builder.dim,
        node_feature_builder=base_feature_builder, edge_feature_builder=graph_edge_builder.encode,
        beam_width=FINAL_BEAM_WIDTH, device=device,
    )
    test_search_metrics = evaluate_puzzles(test_harness, test_puzzles)
    print("Final test solve rate:", test_search_metrics["solve_rate"])
    del test_harness
model = model.to("cpu")


[I 2026-10-18 16:05:46,768] A new study created in memory with name: no-name-c7943f92-c25a-49c3-b3f4-fde8c7efb2be


  0%|          | 0/12 [00:00<?, ?it/s]

/mnt/code/python_projects/nyt-connections-solver/.venv/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Epoch 1/25 | Loss: 1.3690 | Val mean state NDCG: 0.0110 | LR: 8.8e-05 | Train: 3.8s | Validation: 0.3s
Epoch 2/25 | Loss: 1.3260 | Val mean state NDCG: 0.0199 | LR: 8.8e-05 | Train: 3.5s | Validation: 0.3s
Epoch 3/25 | Loss: 1.2982 | Val mean state NDCG: 0.0153 | LR: 8.8e-05 | Train: 3.1s | Validation: 0.3s
Epoch 4/25 | Loss: 1.3389 | Val mean state NDCG: 0.0226 | LR: 8.8e-05 | Train: 3.5s | Validation: 0.3s
Epoch 5/25 | Loss: 1.3669 | Val mean state NDCG: 0.0092 | LR: 8.8e-05 | Train: 3.8s | Validation: 0.3s
Epoch 6/25 | Loss: 1.2890 | Val mean state NDCG: 0.0144 | LR: 8.8e-05 | Train: 2.9s | Validation: 0.3s
Epoch 7/25 | Loss: 1.3424 | Val mean state NDCG: 0.0129 | LR: 4.4e-05 | Train: 3.3s | Validation: 0.3s
Epoch 8/25 | Loss: 1.2966 | Val mean state NDCG: 0.0143 | LR: 4.4e-05 | Train: 2.8s | Validation: 0.3s
Epoch 9/25 | Loss: 1.3447 | Val mean state NDCG: 0.0160 | LR: 4.4e-05 | Train: 3.4s | Validation: 0.3s
Epoch 10/25 | Loss: 1.3231 | Val mean state NDCG: 0.0210 | LR: 2.2e-05 | 

KeyboardInterrupt: 

# Optional node-feature ablation
Compare text-only and text-plus-spelling nodes with matched seeds and the same
edge channels. Only validation ranking metrics are inspected here.


In [ ]:
def run_feature_ablation(epochs=20, seed=0):
    results = []
    for use_spelling in (False, True):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        nodes = TextSpellingFeatures(base_feature_builder.embeddings, use_spelling=use_spelling)
        loaders = {}
        for split, dates_for_split in (("train", train_dates), ("val", val_dates)):
            _, loaders[split] = make_state_loaders(
                connections_df.loc[connections_df["date"].isin(dates_for_split)],
                word_embedder, vectorizer, BATCH_SIZE, shuffle=(split == "train"),
                feature_builder=nodes, edge_providers=graph_edge_builder,
            )
        config = dict(best_model_config, in_dim=nodes.dim)
        candidate = GraphSetTransformer(group_idx=state_artifacts[4]["idx"], **config)
        candidate = train_mixed_states(
            candidate, loaders["train"], loaders["val"], state_artifacts,
            epochs=epochs, device=device, lr=best_training_config["lr"],
            weight_decay=best_training_config["weight_decay"],
        )
        metrics = evaluate_by_state_size(candidate, loaders["val"], cpu_artifacts, ranking_loss, cpu_device)
        results.append({"spelling": use_spelling, **metrics["weighted"]})
    return pd.DataFrame(results)


In [ ]:
RUN_FEATURE_ABLATION = False
if RUN_FEATURE_ABLATION:
    feature_ablation_results = run_feature_ablation()
    display(feature_ablation_results)


# Live inference
After saving, use `load_graph_harness(weights_path)` to reconstruct the frozen
node features and providers. Existing sense-based checkpoints are incompatible
with this node configuration. Feature caching is shared with training; the first
unseen board can be slower than subsequent calls.


# Save Graph Set Transformer Weights

Save the graph checkpoint and feature metadata. Reload with load_graph_harness above; the standard CLI does not load notebook-local graph classes.

In [ ]:
from datetime import datetime
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
weights_path = weights_dir / f"graph_set_transformer_optuna_text_spelling_{best_training_config['relevance_mode']}_{timestamp}.pt"

model_config = best_model_config.copy()
split_dates = {
    split_name: pd.to_datetime(split).strftime("%Y-%m-%d").tolist()
    for split_name, split in (("train", train_dates), ("val", val_dates), ("test", test_dates))
}
checkpoint = {
    "model_class": "GraphSetTransformer",
    "edge_feature_names": list(graph_edge_builder.feature_names),
    "edge_context": "original_16_words",
    "model_config": model_config,
    "training_config": best_training_config,
    "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "embedding_model_name": EMBEDDING_MODEL,
    "feature_metadata": feature_metadata,
    "split_dates": split_dates,
    "dataset_path": str(DATASET_PATH.resolve()),
}
torch.save(checkpoint, weights_path)
print(f"Saved Graph Set Transformer checkpoint to {weights_path}")